## Linear orchestrator (PCA / LN / penalized regressions)

**Run order:** **Imports** → **Configuration** → **Load data** → **Model registry** → **Run**.

1. **Imports** — paths, third-party packages, `utils` / `models` (reloads `models.linear`).
2. **Configuration** — dataset, sample, `REVISED_MACRO`, horizons, `RESULTS_CSV` (uses `REPO_ROOT` from Imports).
3. **Load data** — builds `X`, `xr`, `dates`.
4. **Model registry** — `build_model_factories` + `SELECTED_MODELS`.
5. **Run** — tqdm, OOS R², RSZ line, CSV.

**RSZ / `rsz_pvalue`:** `utils.base_utils.RSZ_Signif` (Bianchi et al., 2021 replication style). Compares OOS forecasts to the **expanding historical mean** of the target (`gap` aligns information timing; rows with NaN forecasts are excluded). Builds \(f_t\), fits **OLS of \(f_t\)** on **a constant** with **HAC (`maxlags=12`)**. Returns **`1 − F(df)(t)`** for that intercept (**small ⇒** stronger evidence the model beats the historical-mean benchmark on this test). Not a generic p-value for R² alone.

**Macro timing:** If `REVISED_MACRO` is **False**, `expanding_window(realtime=True)` overwrites the **`fred`** block each OOS date (`ForecastVintageMacroStore`). If **True**, macro stays the static shifted FRED panel in `X`.


In [6]:
# Imports & repo path — run this cell before any other code cell.
import importlib
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

_cur = Path.cwd().resolve()
_repo_root = None
for p in [_cur, *_cur.parents]:
    if (p / "models").is_dir() and (p / "utils").is_dir():
        _repo_root = p
        break
if _repo_root is None:
    raise RuntimeError(
        "Cannot find repo root (need folders models/ and utils/). cwd="
        + str(_cur)
        + ". cd into TIO4900-Replication or start Jupyter from the repo."
    )

REPO_ROOT = str(_repo_root)
REPOROOT = _repo_root

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import utils.base_utils as bu
import utils.window_utils as wu
from utils.macro_grouping import add_group_level, build_full_group_mapping, groups_as_array

import models.linear as _ml  # noqa: E402
importlib.reload(_ml)

from models.base import (
    PCABaselineModel,
    PCAFirst8PCsWithCPModel,
    LudvigsonNgWithCPModel,
)
from models.linear import *

print("Imports ok | REPO_ROOT =", REPO_ROOT)


Imports ok | REPO_ROOT = /home/ulrikts/Documents/NTNU/TIO4900-Replication


In [7]:
# ----------------------------
# Configuration — requires Imports cell (defines REPO_ROOT, pd, np, …)
# ----------------------------
if "REPO_ROOT" not in globals():
    raise RuntimeError("Run the **Imports** cell above first.")

DATASET = "kr"  # lw | kr | gsw
START_DATE = "1971-08-31"
#END_DATE = "2018-12-31"
END_DATE = '2025-06-30'

REVISED_MACRO = False  # True: static shifted FRED-MD in X, expanding_window(realtime=False)
# False: macro in X is only a column template; expanding_window(realtime=True) reloads
#     forecast-vintage macro each OOS date via utils.forecast_vintages.ForecastVintageMacroStore

FREQUENCY = "annual"  # monthly: monthly yields + get_monthly_forward_rates + h=1 xr | annual: yearly + get_forward_rates

HOLDING_MONTHS_ANNUAL = 12
HOLDING_MONTHS_MONTHLY = 1

GAP = 11  # timing vs realized return in expanding_window / RSZ (use 0 for h=1 monthly as in CT-style stack)
OOS_START = pd.Timestamp("1990-01-31")

TARGET_MATURITIES = ["24", "36", "48", "60", "84", "120"]

#TARGET_MATURITIES = ["120"]
CP_COLS = ["24", "36", "48", "60"]

# With FREQUENCY="monthly", get_monthly_forward_rates → 10 cols (12,…,120); FWD_ONLY_N=10 = full curve.
FWD_ONLY_N = 10

MATURITIES_ANNUAL_DEFAULT = [str(i) for i in range(12, 121) if i % 12 == 0]
MATURITIES_MONTHLY_DEFAULT = [str(i) for i in range(1, 121)]

FFILL_PANEL = False

# Evaluation sub-periods — forecasts are produced once over the full sample (to END_DATE),
# then R²/RSZ are reported for each cutoff by truncating the series. The expanding-mean
# benchmark grows from the sample start, so truncating at a cutoff yields exactly the
# OOS metrics for OOS_START..cutoff. Running the 2025 sample therefore also yields 2018.
EVAL_PERIODS = {
    "2018": pd.Timestamp("2018-12-31"),
    "2025": pd.Timestamp(END_DATE),
}

# Winsorization (L&N only) — mirrors monthly_returns_performance.ipynb
WINSORIZE_LN = True
LN_MODEL_NAME = "L&N (2009)"
WINSOR_Q_LOW = 0.001
WINSOR_Q_HIGH = 0.999

RESULTS_CSV = os.path.join(REPO_ROOT, "notebooks", "orchestrators", "results", "linear_runs.csv")
# Long-format per-date forecasts, appended (never overwritten) so R² can be recomputed later.
FORECASTS_CSV = os.path.join(REPO_ROOT, "notebooks", "orchestrators", "results", "linear_forecasts.csv")


In [8]:
# Load data (uses bu, wu, FREQUENCY, DATASET, … from Imports + Configuration)
if "bu" not in globals():
    raise RuntimeError("Run **Imports** then **Configuration**, then this cell.")

if FREQUENCY not in {"annual", "monthly"}:
    raise ValueError('FREQUENCY must be "annual" or "monthly".')
if FREQUENCY == "monthly" and DATASET == "gsw":
    raise ValueError('Monthly xr needs lw/kr monthly yields (not gsw).')

if FREQUENCY == "annual":
    yields = bu.get_yields(
        type=DATASET,
        start=START_DATE,
        end=END_DATE,
        maturities=MATURITIES_ANNUAL_DEFAULT,
    )
    forward = bu.get_forward_rates(yields)
    xr = bu.get_excess_returns(yields, horizon=HOLDING_MONTHS_ANNUAL).dropna()
else:
    yields = bu.get_yields(
        type=DATASET,
        start=START_DATE,
        end=END_DATE,
        maturities=MATURITIES_MONTHLY_DEFAULT,
    )
    forward = bu.get_monthly_forward_rates(yields)
    xr = bu.get_excess_returns(yields, horizon=HOLDING_MONTHS_MONTHLY).dropna()

fred_md_start_date = pd.to_datetime(START_DATE) - pd.DateOffset(months=6)
fred_md_raw = bu.get_fred_data(
    "data/2026-01-MD.csv", start=fred_md_start_date, end=END_DATE
)

monthly_yields = bu.get_yields(
    type=DATASET,
    start=START_DATE,
    end=END_DATE,
    maturities=[str(i) for i in range(1, 121)],
)
monthly_xr = bu.get_excess_returns(monthly_yields, horizon=1).dropna()

fred_md = fred_md_raw.shift(1)
fred_md = fred_md.drop(columns=["TWEXAFEGSMTHx", "ACOGNO"])
fred_md = fred_md.loc[START_DATE:END_DATE]

yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]
fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]
monthly_xr = monthly_xr.loc[monthly_xr.index <= xr.index[-1]]

s2g = build_full_group_mapping(fred_md, forward, yields)

# fred block uses shifted revised FRED-MD for column layout. If REVISED_MACRO is False,
# expanding_window(realtime=True) replaces these values each date with vintages (see Run cell).
X = pd.concat([fred_md, forward, yields], axis=1, keys=["fred", "forward", "yields"])
X = add_group_level(X, s2g, level_name="group")
X = X.sort_index(axis=1, level="group")
groups = groups_as_array(X, level="group")

if FFILL_PANEL:
    X = X.ffill()

dates = xr.index

for m in TARGET_MATURITIES:
    if m not in xr.columns:
        raise KeyError(f"Maturity {m} not in xr columns: got {list(xr.columns[:5])}...")

if X["forward"].shape[1] < FWD_ONLY_N:
    raise ValueError(
        f"Forward block has {X['forward'].shape[1]} columns; need ≥ FWD_ONLY_N={FWD_ONLY_N}. "
        "Use annual lw or reduce FWD_ONLY_N."
    )

print(
    "dataset:",
    DATASET,
    "| xr:",
    xr.shape,
    "| X:",
    X.shape,
    "| forward cols:",
    X["forward"].shape[1],
    "| FWD_ONLY_N:",
    FWD_ONLY_N,
    "| macro in window:",
    "revised static fred" if REVISED_MACRO else "ForecastVintageMacroStore (realtime=True)",
    "| frequency:",
    FREQUENCY,
)

dataset: kr | xr: (635, 9) | X: (635, 144) | forward cols: 10 | FWD_ONLY_N: 10 | macro in window: ForecastVintageMacroStore (realtime=True) | frequency: annual


/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:219: UserWarning: The following series are defined in get_fredmd_grouping() but are not present in the FRED-MD data: ['ACOGNO', 'TWEXAFEGSMTHx']. They may have been dropped or renamed.
  warnings.warn(
/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:168: UserWarning: 2 entries in series_to_group are not present in the DataFrame columns: ['ACOGNO', 'TWEXAFEGSMTHx']
  warnings.warn(


In [9]:
if "RegularizedLinearModel" not in globals():
    raise RuntimeError("Run **Imports** cell first (defines model classes).")
if "FWD_ONLY_N" not in globals():
    raise RuntimeError("Run **Configuration** before this cell (defines FWD_ONLY_N).")


def winsorize_series_expanding(y_hat, q_low=0.001, q_high=0.999):
    """Expanding-window winsorization for a single forecast series (strictly-past).

    At each time t the clip bounds are computed from y_hat[:t] — the strictly past
    history, NOT including time t itself. This ensures an outlier at t is clamped
    to 'the max/min last seen'.
    """
    y_hat = np.asarray(y_hat, dtype=float)
    T = len(y_hat)
    out = np.full(T, np.nan)

    for t in range(T):
        v = y_hat[t]
        if not np.isfinite(v):
            continue
        hist = y_hat[:t]  # strictly past (excludes t)
        finite_hist = hist[np.isfinite(hist)]
        if finite_hist.size == 0:
            out[t] = v  # no history yet — pass through
            continue
        lo = np.nanquantile(finite_hist, q_low)
        hi = np.nanquantile(finite_hist, q_high)
        out[t] = np.clip(v, lo, hi)

    return out


def build_model_factories(xr_panel, cp_cols):
    """Return factories that build a fresh estimator per expanding-window template."""
    return {
        "PCA(8)": lambda: PCAFirst8PCsWithCPModel(
            xr_full=xr_panel,
            cp_cols=cp_cols,
            components=8,
            n_cp_forwards=5,
        ),
        "L&N (2009)": lambda: LudvigsonNgWithCPModel(
            xr_full=xr_panel,
            cp_cols=cp_cols,
            n_factors=8,
            n_cp_forwards=5,
        ),
        "Ridge CP + macro (direct)": lambda: RidgeRawMacroCPModel(
            xr_full=xr_panel,
            cp_cols=cp_cols,
            macro_series="fred",
            forward_series="forward",
        ),
        "Lasso CP + macro (direct)": lambda: LassoRawMacroCPModel(
            xr_full=xr_panel,
            cp_cols=cp_cols,
            macro_series="fred",
            forward_series="forward",
        ),
        "ElasticNet CP + macro (direct)": lambda: ElasticNetRawMacroCPModel(
            xr_full=xr_panel,
            cp_cols=cp_cols,
            macro_series="fred",
            forward_series="forward",
        ),
        f"Ridge {FWD_ONLY_N} forwards + macro (direct)": lambda: RidgeRawMacroFwdDirectModel(
            macro_series="fred",
            forward_series="forward",
            forward_max_cols=FWD_ONLY_N,
        ),
        f"Lasso {FWD_ONLY_N} forwards + macro (direct)": lambda: LassoRawMacroFwdDirectModel(
            macro_series="fred",
            forward_series="forward",
            forward_max_cols=FWD_ONLY_N,
        ),
        f"ElasticNet {FWD_ONLY_N} forwards + macro (direct)": lambda: ElasticNetRawMacroFwdDirectModel(
            macro_series="fred",
            forward_series="forward",
            forward_max_cols=FWD_ONLY_N,
        ),
        # Forward rates only (no macro): RegularizedLinearModel, first FWD_ONLY_N cols of X['forward']
        f"ElasticNet fwd-only ({FWD_ONLY_N})": lambda: RegularizedLinearModel(
            model_type="elasticnet",
            include_fred=False,
            forward_max_cols=FWD_ONLY_N,
        ),
        f"Lasso fwd-only ({FWD_ONLY_N})": lambda: RegularizedLinearModel(
            model_type="lasso",
            include_fred=False,
            forward_max_cols=FWD_ONLY_N,
        ),
        f"Ridge fwd-only ({FWD_ONLY_N})": lambda: RegularizedLinearModel(
            model_type="ridge",
            include_fred=False,
            forward_max_cols=FWD_ONLY_N,
        ),
        # PCR on first FWD_ONLY_N forward rates only (no macro): PCABaselineModel in base.py
        f"PCR forward ({FWD_ONLY_N} fwds, 3 PCs)": lambda: PCABaselineModel(
            components=3,
            series="forward",
            max_cols=FWD_ONLY_N,
        ),
    }


SELECTED_MODELS = [
    #f"PCR forward ({FWD_ONLY_N} fwds, 3 PCs)",
    # "PCA(8)",  # excluded per monthly forward-rate run
    #"L&N (2009)",
    #f"ElasticNet {FWD_ONLY_N} forwards + macro (direct)",
    f"ElasticNet fwd-only ({FWD_ONLY_N})",
]


In [10]:
if "build_model_factories" not in globals():
    raise RuntimeError("Run Imports → Configuration → Load → Model registry, then this cell.")

_factories = build_model_factories(xr, CP_COLS)
for k in SELECTED_MODELS:
    if k not in _factories:
        raise KeyError(f"No factory for '{k}'. Available: {sorted(_factories)}")

records = []
forecast_records = []
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

def _make_row(name, maturity, r2, signif, winsorized, eval_period, eval_end):
    return {
        "run_id": RUN_ID,
        "time_now": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "model": name,
        "maturity": maturity,
        "eval_period": eval_period,
        "eval_end": pd.Timestamp(eval_end).strftime("%Y-%m-%d"),
        "r2_oos": r2,
        "rsz_pvalue": signif,
        "dataset": DATASET,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "revised_macro_static_X": REVISED_MACRO,
        "expanding_window_realtime": not REVISED_MACRO,
        "frequency": FREQUENCY,
        "gap": GAP,
        "winsorized": winsorized,
    }

def _collect_forecasts(name, maturity, winsorized, y_true, y_hat):
    # Store the full per-date series (in-sample y_hat is NaN). Keeping every date — not just
    # the OOS region — lets the expanding historical-mean benchmark (and thus R²) be
    # recomputed afterward for any gap.
    for d, yt, yh in zip(dates, y_true, y_hat):
        forecast_records.append({
            "run_id": RUN_ID,
            "model": name,
            "maturity": maturity,
            "date": pd.Timestamp(d).strftime("%Y-%m-%d"),
            "y_true": float(yt) if np.isfinite(yt) else np.nan,
            "y_hat": float(yh) if np.isfinite(yh) else np.nan,
            "winsorized": winsorized,
            "dataset": DATASET,
            "frequency": FREQUENCY,
            "gap": GAP,
            "oos_start": str(OOS_START),
            "revised_macro_static_X": REVISED_MACRO,
            "expanding_window_realtime": not REVISED_MACRO,
        })

# Number of dates falling in each eval sub-period (truncation length, from sample start).
_dates_idx = pd.to_datetime(pd.Index(dates))
_period_len = {label: int((_dates_idx <= cut).sum()) for label, cut in EVAL_PERIODS.items()}

def _score_periods(name, maturity, y, y_hat, winsorized, tag=""):
    # Score every eval sub-period from a single full-sample forecast series by truncating
    # at each cutoff. The expanding benchmark grows from the start, so the truncated metric
    # equals the OOS metric for OOS_START..cutoff.
    for label, cut in EVAL_PERIODS.items():
        idx = _period_len[label]
        ys, yhs = y[:idx], y_hat[:idx]
        r2 = float(wu.oos_r2(ys, yhs, gap=GAP))
        signif = bu.RSZ_Signif(ys, yhs, gap=GAP)
        records.append(_make_row(name, maturity, r2, signif, winsorized, label, cut))
        print(f"    {tag}[{label}] OOS R² = {r2:.6f} | RSZ = {signif}")

print(f"\nScheduled {len(SELECTED_MODELS)} models × {len(TARGET_MATURITIES)} maturities × {len(EVAL_PERIODS)} periods")
print(f"eval periods: {{ {', '.join(f'{k}<={pd.Timestamp(v).date()}' for k,v in EVAL_PERIODS.items())} }}")
print(f"expanding_window(realtime={not REVISED_MACRO})\n")

for name in tqdm(SELECTED_MODELS, desc="model"):
    for maturity in TARGET_MATURITIES:
        print(f"\n>>> {name} | maturity={maturity} ({FREQUENCY})")
        tmpl = _factories[name]()
        y = xr[maturity].values
        y_hat = wu.expanding_window(
            tmpl,
            X,
            y,
            dates,
            OOS_START,
            gap=GAP,
            refit_freq=1,
            coef_callback=None,
            progress=False,
            realtime=not REVISED_MACRO,
        )

        # Raw results — scored for every eval sub-period from this single forecast series
        _collect_forecasts(name, maturity, False, y, y_hat)
        _score_periods(name, maturity, y, y_hat, winsorized=False)

        # L&N only: winsorize this maturity's forecasts immediately (per-maturity, strictly-past)
        if WINSORIZE_LN and name == LN_MODEL_NAME:
            y_hat_win = winsorize_series_expanding(y_hat, q_low=WINSOR_Q_LOW, q_high=WINSOR_Q_HIGH)
            _collect_forecasts(f"{LN_MODEL_NAME} (winsor)", maturity, True, y, y_hat_win)
            _score_periods(f"{LN_MODEL_NAME} (winsor)", maturity, y, y_hat_win, winsorized=True, tag="[winsor] ")

out_df = pd.DataFrame(records)
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
# Append-only: each run adds rows (tagged by run_id) instead of overwriting prior results.
out_df.to_csv(RESULTS_CSV, mode="a", header=not os.path.exists(RESULTS_CSV), index=False)

fc_df = pd.DataFrame(forecast_records)
fc_df.to_csv(FORECASTS_CSV, mode="a", header=not os.path.exists(FORECASTS_CSV), index=False)

print(f"\n=== Appended {len(out_df)} summary rows to {RESULTS_CSV} (run_id={RUN_ID}) ===")
print(f"=== Appended {len(fc_df)} forecast rows to {FORECASTS_CSV} (run_id={RUN_ID}) ===")
print(out_df.to_string(index=False))


Scheduled 1 models × 6 maturities × 2 periods
eval periods: { 2018<=2018-12-31, 2025<=2025-06-30 }
expanding_window(realtime=True)



model:   0%|          | 0/1 [00:00<?, ?it/s]


>>> ElasticNet fwd-only (10) | maturity=24 (annual)
    [2018] OOS R² = -0.145565 | RSZ = 0.9718095373288347
    [2025] OOS R² = -0.085000 | RSZ = 0.7727442776273219

>>> ElasticNet fwd-only (10) | maturity=36 (annual)
    [2018] OOS R² = -0.174974 | RSZ = 0.7807797521661386
    [2025] OOS R² = -0.138707 | RSZ = 0.7368104457933907

>>> ElasticNet fwd-only (10) | maturity=48 (annual)
    [2018] OOS R² = -0.169397 | RSZ = 0.7272201209158717
    [2025] OOS R² = -0.155228 | RSZ = 0.7684061665506716

>>> ElasticNet fwd-only (10) | maturity=60 (annual)
    [2018] OOS R² = -0.167377 | RSZ = 0.7621373030273251
    [2025] OOS R² = -0.169770 | RSZ = 0.8401411844897171

>>> ElasticNet fwd-only (10) | maturity=84 (annual)
    [2018] OOS R² = -0.220752 | RSZ = 0.866463050524566
    [2025] OOS R² = -0.225111 | RSZ = 0.9165582089035887

>>> ElasticNet fwd-only (10) | maturity=120 (annual)


model: 100%|██████████| 1/1 [14:50<00:00, 890.96s/it]

    [2018] OOS R² = -0.223278 | RSZ = 0.8018237786546997
    [2025] OOS R² = -0.226545 | RSZ = 0.8517900811540139

=== Appended 12 summary rows to /home/ulrikts/Documents/NTNU/TIO4900-Replication/notebooks/orchestrators/results/linear_runs.csv (run_id=20260604_085253) ===
=== Appended 3810 forecast rows to /home/ulrikts/Documents/NTNU/TIO4900-Replication/notebooks/orchestrators/results/linear_forecasts.csv (run_id=20260604_085253) ===
         run_id            time_now                    model maturity eval_period   eval_end    r2_oos  rsz_pvalue dataset start_date   end_date  revised_macro_static_X  expanding_window_realtime frequency  gap  winsorized
20260604_085253 2026-06-04 08:55:17 ElasticNet fwd-only (10)       24        2018 2018-12-31 -0.145565    0.971810      kr 1971-08-31 2025-06-30                   False                       True    annual   11       False
20260604_085253 2026-06-04 08:55:17 ElasticNet fwd-only (10)       24        2025 2025-06-30 -0.085000    0.772744 